# Comparación numérica entre EpiEstim en R y la implementación Python

Comprobamos tres etapas de la ruta `parametric_si` para **incidencia agregada por semana**: (1) los pesos diarios calculados por [`EpiEstim::discr_si()`](https://mrc-ide.github.io/EpiEstim/reference/discr_si.html), (2) la incidencia diaria reconstruida con [el algoritmo de EpiEstim para datos agregados](https://mrc-ide.github.io/EpiEstim/articles/EpiEstim_aggregated_data.html), y (3) la media y los límites 95 % de la posterior de `R_t` al final de cada semana. La referencia R se ejecuta mediante [este script](../scripts/compare_epiestim_reference.R); Python usa [esta implementación](../scripts/epiestim_parametric_weekly.py).

La ejecución guardada usó **EpiEstim 2.4** de [este commit](https://github.com/mrc-ide/EpiEstim/tree/31172df212c22d8b0e73fdfa0643c423988eb452). Para repetirla se necesita R con `EpiEstim` y las dependencias Python del proyecto. El ejecutable se detecta como `Rscript`; si se usa otro, puede señalarse con `EPIESTIM_R_EXEC`. El cuaderno muestra la versión de R realmente ejecutada.

## 1. Datos y parámetros compartidos

Leemos el [CSV semanal nacional](../docs/data_processed/evolucion_semanal/weekly_series_2007_2025.csv), derivado de los registros individuales de Sivigila por `data-processing`. Usamos las primeras 20 semanas consecutivas de 2009, 2015 y 2023 para cubrir diferentes niveles de incidencia. Son **notificaciones por semana**, no infecciones observadas. Ambos programas reciben exactamente el mismo vector y usan intervalo serial de media 14,1 y DE 2,4 días, previa Gamma de media 5 y DE 5, `dt=7`, ventana `dt_out=14`, `recon_opt='naive'` y 10 iteraciones.

In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'scripts/epiestim_parametric_weekly.py').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'scripts/epiestim_parametric_weekly.py').is_file()
sys.path.insert(0, str(ROOT / 'scripts'))
from epiestim_parametric_weekly import discr_si, estimate_weekly

R_EXEC = os.environ.get('EPIESTIM_R_EXEC') or shutil.which('Rscript')
if not R_EXEC:
    raise RuntimeError('Instale R y EpiEstim, o defina EPIESTIM_R_EXEC con la ruta de R/Rscript.')
print('Ejecutable R:', R_EXEC)
weekly = pd.read_csv(ROOT / 'docs/data_processed/evolucion_semanal/weekly_series_2007_2025.csv')
bloques = {}
for year in (2009, 2015, 2023):
    part = weekly.loc[(weekly.year == year) & (weekly.age_group == 'all') & weekly.week.between(1, 20)].sort_values('week')
    assert len(part) == 20 and part.week.tolist() == list(range(1, 21))
    assert part.notifications.notna().all()
    bloques[year] = part.notifications.to_numpy(dtype=int)
display(pd.DataFrame([{'año': y, 'semanas': len(c), 'mínimo semanal': c.min(), 'máximo semanal': c.max()} for y, c in bloques.items()]))

Ejecutable R: /tmp/r-local/usr/bin/R


,año,semanas,mínimo semanal,máximo semanal
0,2009,20,1133,1931
1,2015,20,1450,2546
2,2023,20,302,694


## 2. Ejecución de EpiEstim y comparación

R produce estimaciones para ventanas que terminan en distintos días. La salida publicada por Python corresponde a **fines de semana**: filtramos las filas de R con `t_end` múltiplo de 7 y alineamos la semana `t_end/7`. Los pesos de R incluyen el rezago 0 y un último día adicional al horizonte de incidencia; Python evalúa `discr_si()` en esos mismos rezagos. Comparamos solo estimaciones presentes en ambas salidas.

In [2]:
def comando_r(executable, script, input_csv, output_prefix):
    if Path(executable).name == 'Rscript':
        return [executable, str(script), str(input_csv), str(output_prefix)]
    return [executable, '--vanilla', '--slave', '-f', str(script), '--args', str(input_csv), str(output_prefix)]

resultados = []
versiones = {}
for year, counts in bloques.items():
    with TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        input_csv = tmp / 'weekly.csv'
        prefix = tmp / 'r_reference'
        pd.DataFrame({'notifications': counts}).to_csv(input_csv, index=False)
        run = subprocess.run(comando_r(R_EXEC, ROOT / 'scripts/compare_epiestim_reference.R', input_csv, prefix), cwd=ROOT, text=True, capture_output=True)
        if run.returncode:
            raise RuntimeError(f'EpiEstim falló para {year}:\n{run.stdout}\n{run.stderr}')
        versiones[year] = (tmp / 'r_reference_versions.txt').read_text().strip()
        r_weights = pd.read_csv(tmp / 'r_reference_weights.csv')
        r_daily = pd.read_csv(tmp / 'r_reference_daily.csv')
        r_post = pd.read_csv(tmp / 'r_reference_posterior.csv')
        py = estimate_weekly(counts, 14.1, 2.4, window_weeks=2, prior_mean=5, prior_sd=5, iterations=10)
        py_weights = discr_si(r_weights.lag.to_numpy(), 14.1, 2.4)
        r_weekly = r_post.loc[r_post.t_end.mod(7).eq(0)].copy()
        indices = r_weekly.t_end.to_numpy(dtype=int) // 7 - 1
        assert np.array_equal(r_weekly.t_start.to_numpy(dtype=int), r_weekly.t_end.to_numpy(dtype=int) - 13)
        assert np.isfinite(py.estimates[indices]).all()
        assert len(indices) == np.isfinite(py.estimates[:, 0]).sum()
        assert np.allclose(py.daily_incidence.reshape(-1, 7).sum(axis=1), counts)
        r_summaries = r_weekly[['Mean(R)', 'Quantile.0.025(R)', 'Quantile.0.975(R)']].to_numpy()
        errors = np.abs(r_summaries - py.estimates[indices])
        resultados.append({'año': year, 'ventanas comparadas': len(indices), 'máx. diferencia pesos': np.max(abs(r_weights.weight - py_weights)), 'máx. diferencia incidencia diaria': np.max(abs(r_daily.reconstructed.to_numpy() - py.daily_incidence)), 'máx. diferencia media Rt': errors[:, 0].max(), 'máx. diferencia límite inferior': errors[:, 1].max(), 'máx. diferencia límite superior': errors[:, 2].max()})

tabla = pd.DataFrame(resultados)
display(tabla)
print('Versiones de referencia:')
print(versiones[2015])
assert (tabla['máx. diferencia pesos'] < 1e-10).all()
assert (tabla['máx. diferencia incidencia diaria'] < 1e-8).all()
assert (tabla['máx. diferencia media Rt'] < 1e-10).all()
assert (tabla['máx. diferencia límite inferior'] < 1e-10).all()
assert (tabla['máx. diferencia límite superior'] < 1e-10).all()

,año,ventanas comparadas,máx. diferencia pesos,máx. diferencia incidencia diaria,máx. diferencia media Rt,máx. diferencia límite inferior,máx. diferencia límite superior
0,2009,17,2.001177e-14,1.080025e-12,5.995204e-15,8.659740e-15,8.437695e-15
1,2015,17,2.001177e-14,1.364242e-12,7.327472e-15,7.771561e-15,9.325873e-15
2,2023,17,2.001177e-14,6.536993e-13,8.437695e-15,5.329071e-15,7.993606e-15


Versiones de referencia:
R: R version 4.3.3 (2024-02-29)
EpiEstim: 2.4
Weekly counts: 20
dt: 7; dt_out: 14; recon_opt: naive; iter: 10
SI mean: 14.1; SI SD: 2.4; prior mean: 5; prior SD: 5


## Alcance

Esta es una comprobación **numérica de implementación** en tres tramos sin semanas faltantes. No valida los parámetros seriales para Colombia, la elección de previa, la reconstrucción diaria como descripción de las fechas reales ni el modelo epidemiológico. La serie diaria obtenida directamente de `FEC_NOT` es una alternativa para un análisis de sensibilidad separado; esta comparación mantiene la entrada semanal porque verifica el procedimiento publicado.